# FraudeCero · Grupo 6 · 02b · Optimización (A2.6)
**Proyecto Integrador Big Data – Uniciencia 2026-II** · Integrantes: Juan David Montoya · William Conde Villamizar


**Objetivo:** aplicar técnicas de optimización y medir el **tiempo antes y después** de cada una con la misma consulta.

| Técnica | Consulta de prueba |
|---|---|
| 1. **Particionamiento** por `hora` | Fraudes de la franja 0–5 h |
| 2. **OPTIMIZE + Z-ORDER** por `Amount` (Delta) | Transacciones con monto > 1.000 EUR |
| 3. **Caché** en memoria | Agregación por hora y rango de monto |
| 4. **Broadcast join** | Unión SFC × KPI de fraude |

Cada medición es la **mediana de 3 ejecuciones** (después de 1 ejecución de calentamiento) para reducir el ruido.

## Configuración común
Esta celda es igual en todos los notebooks. Detecta si se ejecuta en **Google Colab** o en **Jupyter / GitHub Actions**, instala PySpark 3.5 + Delta Lake (OSS) y define:
- `LAKE`: carpeta del Lakehouse (capas `bronce`, `plata`, `oro`).
- `EVID`: carpeta `99_evidencias` (tablas de evidencia, log de monitoreo, tableros HTML).
- `guardar()` / `leer()`: escriben y leen tablas **Delta** (si Delta no está disponible usan Parquet y lo avisan).
- `registrar()`: escribe en el log de monitoreo el tiempo, las filas, el % de nulos y la memoria de cada etapa.

> En Colab cada notebook corre en una máquina distinta, por eso el Lakehouse se respalda en Google Drive al final de cada notebook y se restaura al inicio del siguiente.

In [ ]:
# ===================== CONFIGURACIÓN COMÚN FRAUDECERO =====================
NOTEBOOK = "02b_optimizacion"
import os, sys, time, json, shutil, datetime, subprocess, hashlib
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub", "psutil", "plotly"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DRIVE = "/content/drive/MyDrive/FraudeCero"
    LAKE = "/content/lakehouse"                      # disco local (rápido)
    LAKE_RESPALDO = f"{BASE_DRIVE}/lakehouse"        # respaldo persistente en Drive
    EVID = f"{BASE_DRIVE}/99_evidencias"
else:
    RAIZ = os.environ.get("FRAUDECERO_RAIZ", os.path.abspath(".."))
    LAKE = os.environ.get("FRAUDECERO_LAKE", os.path.join(RAIZ, "lakehouse"))
    LAKE_RESPALDO = None
    EVID = os.environ.get("FRAUDECERO_EVID", os.path.join(RAIZ, "99_evidencias"))

for carpeta in [LAKE, EVID, f"{EVID}/monitoreo", f"{EVID}/tablas"]:
    os.makedirs(carpeta, exist_ok=True)

def restaurar_lake():
    """En Colab: trae el Lakehouse desde Drive si el disco local está vacío."""
    if LAKE_RESPALDO and os.path.exists(LAKE_RESPALDO) and not os.listdir(LAKE):
        shutil.copytree(LAKE_RESPALDO, LAKE, dirs_exist_ok=True)
        print("Lakehouse restaurado desde Drive")

def respaldar_lake():
    """En Colab: copia el Lakehouse a Drive (sin la zona landing de archivos crudos)."""
    if LAKE_RESPALDO:
        shutil.copytree(LAKE, LAKE_RESPALDO, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("landing"))
        print("Lakehouse respaldado en Drive:", LAKE_RESPALDO)

restaurar_lake()

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

FORMATO = "delta"
try:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
               .config("spark.driver.memory", "6g")
               .config("spark.sql.shuffle.partitions", "8")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.range(1).write.format("delta").mode("overwrite").save(f"{LAKE}/_prueba_delta")
except Exception as e:
    print("⚠️ Delta Lake no disponible, se usará Parquet. Detalle:", str(e)[:300])
    FORMATO = "parquet"
    spark = (SparkSession.builder.appName(f"FraudeCero-{NOTEBOOK}")
             .config("spark.driver.memory", "6g")
             .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ruta(capa, tabla):
    return f"{LAKE}/{capa}/{tabla}"

def leer(capa, tabla):
    df = spark.read.format(FORMATO).load(ruta(capa, tabla))
    df.createOrReplaceTempView(f"{capa}_{tabla}")
    return df

def guardar(df, capa, tabla, particion=None):
    w = df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true")
    if particion:
        w = w.partitionBy(particion)
    w.save(ruta(capa, tabla))
    return leer(capa, tabla)

# ---------- Monitoreo: tiempo, filas, % nulos y memoria por etapa ----------
import psutil, pandas as pd
LOG_MON = f"{EVID}/monitoreo/log_ejecuciones.csv"
ARCHIVO_RUN = f"{EVID}/monitoreo/run_actual.txt"
RUN_ID = os.environ.get("FRAUDECERO_RUN_ID")
if not RUN_ID:
    if NOTEBOOK == "00_ingesta" or not os.path.exists(ARCHIVO_RUN):
        RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")   # cada corrida del pipeline empieza en 00
        with open(ARCHIVO_RUN, "w") as f:
            f.write(RUN_ID)
    else:
        RUN_ID = open(ARCHIVO_RUN).read().strip()
ORIGEN = "github_actions" if os.environ.get("GITHUB_ACTIONS") else ("colab" if EN_COLAB else "jupyter")

def memoria_mb():
    py = psutil.Process(os.getpid()).memory_info().rss / 1024**2
    try:
        rt = spark.sparkContext._jvm.java.lang.Runtime.getRuntime()
        jvm = (rt.totalMemory() - rt.freeMemory()) / 1024**2
    except Exception:
        jvm = 0.0
    return round(py + jvm, 1)

def registrar(etapa, t0, df=None, filas=None):
    """Registra una etapa en el log de monitoreo. t0 = time.time() al inicio de la etapa."""
    duracion = round(time.time() - t0, 2)
    pct_nulos = None
    if df is not None:
        filas = df.count()
        cols = df.columns
        nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).collect()[0]
        pct_nulos = round(sum(v or 0 for v in nulos) / (filas * len(cols)) * 100, 4) if filas else 0.0
    fila = {"run_id": RUN_ID, "origen": ORIGEN, "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "notebook": NOTEBOOK, "etapa": etapa, "duracion_s": duracion, "filas": filas,
            "pct_nulos": pct_nulos, "memoria_mb": memoria_mb(), "cpu_pct": psutil.cpu_percent(interval=0.2),
            "formato": FORMATO}
    pd.DataFrame([fila]).to_csv(LOG_MON, mode="a", index=False, header=not os.path.exists(LOG_MON))
    print(f"⏱️ {etapa}: {duracion} s · filas={filas} · nulos={pct_nulos}% · mem={fila['memoria_mb']} MB")

def mostrar(df, n=20):
    """display() de Databricks no existe en Jupyter/Colab: se muestra como tabla pandas."""
    from IPython.display import display
    display(df.limit(n).toPandas() if hasattr(df, "toPandas") else df)

print(f"Notebook: {NOTEBOOK} · Entorno: {ORIGEN} · Formato de tablas: {FORMATO.upper()} · Spark {spark.version}")
print("LAKE =", LAKE, "| EVID =", EVID)

In [ ]:
import statistics
plata_cc = leer("plata", "transacciones")
plata_sfc = leer("plata", "sfc_tarjetas")
resultados = []

def medir(funcion, repeticiones=3):
    funcion()                                   # calentamiento
    tiempos = []
    for _ in range(repeticiones):
        t = time.time(); funcion(); tiempos.append(time.time() - t)
    return round(statistics.median(tiempos), 3)

def anotar(tecnica, antes, despues, detalle):
    mejora = round((antes - despues) / antes * 100, 1) if antes else None
    resultados.append({"tecnica": tecnica, "tiempo_antes_s": antes, "tiempo_despues_s": despues,
                       "mejora_pct": mejora, "detalle": detalle})
    print(f"{tecnica:<28} antes {antes:>7.3f} s → después {despues:>7.3f} s  ({mejora:+.1f} %)")

### Técnica 1 · Particionamiento por hora
Con la tabla particionada, Spark solo lee las carpetas `hora=0` … `hora=5` (*partition pruning*).

In [ ]:
t0 = time.time()
consulta_1 = lambda df: df.filter(F.col("hora").between(0, 5)).agg(F.sum("Class"), F.sum("Amount")).collect()
antes = medir(lambda: consulta_1(leer("plata", "transacciones")))
guardar(plata_cc, "plata", "transacciones_part_hora", particion="hora")
despues = medir(lambda: consulta_1(leer("plata", "transacciones_part_hora")))
anotar("Particionamiento (hora)", antes, despues, "Tabla plata reescrita con partitionBy('hora')")
registrar("opt_particionamiento", t0)

### Técnica 2 · OPTIMIZE + Z-ORDER
Compacta archivos pequeños y ordena los datos por `Amount`, de modo que los filtros por monto saltan archivos completos (*data skipping*). Requiere Delta Lake.

In [ ]:
t0 = time.time()
ruta_z = ruta("plata", "transacciones_zorder")
# se fuerza una tabla con muchos archivos pequeños para que el efecto sea visible
guardar(plata_cc.repartition(64), "plata", "transacciones_zorder")
consulta_2 = lambda: spark.read.format(FORMATO).load(ruta_z).filter("Amount > 1000").agg(F.count("*"), F.sum("Class")).collect()
antes = medir(consulta_2)
if FORMATO == "delta":
    spark.sql(f"OPTIMIZE delta.`{ruta_z}` ZORDER BY (Amount)").show(truncate=False)
    detalle = "OPTIMIZE delta ZORDER BY (Amount) sobre 64 archivos"
else:
    plata_cc.repartition(4).sortWithinPartitions("Amount").write.format(FORMATO).mode("overwrite").save(ruta_z)
    detalle = "Parquet: compactación a 4 archivos + orden por Amount (equivalente sin Delta)"
despues = medir(consulta_2)
anotar("OPTIMIZE + Z-ORDER (Amount)", antes, despues, detalle)
registrar("opt_zorder", t0)

### Técnica 3 · Caché
La tabla queda en memoria tras la primera lectura y las consultas repetidas (típicas del tablero) no vuelven a leer disco.

In [ ]:
t0 = time.time()
consulta_3 = lambda df: df.groupBy("hora", "rango_monto").agg(F.count("*"), F.sum("Class"), F.avg("Amount")).collect()
base = leer("plata", "transacciones")
antes = medir(lambda: consulta_3(base))
cacheada = leer("plata", "transacciones").cache()
cacheada.count()
despues = medir(lambda: consulta_3(cacheada))
cacheada.unpersist()
anotar("Caché en memoria", antes, despues, "df.cache() antes de consultas repetidas")
registrar("opt_cache", t0)

### Técnica 4 · Broadcast join
La tabla de KPI tiene 1 fila: se envía completa a cada ejecutor y se evita el *shuffle* del join.

In [ ]:
t0 = time.time()
kpi = plata_cc.groupBy("producto").agg(F.avg("Class").alias("tasa_fraude"))
union = lambda k: plata_sfc.join(k, "producto").agg(F.sum(F.col("valor") * F.col("tasa_fraude"))).collect()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)       # antes: join con shuffle (sort-merge)
antes = medir(lambda: union(kpi))
despues = medir(lambda: union(F.broadcast(kpi)))                 # después: broadcast hash join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
anotar("Broadcast join", antes, despues, "F.broadcast() sobre la tabla de KPI (1 fila)")
registrar("opt_broadcast", t0)

## Tabla de tiempos antes / después (evidencia A2.6)

In [ ]:
tabla_opt = pd.DataFrame(resultados)
tabla_opt.to_csv(f"{EVID}/tablas/optimizacion_tiempos.csv", index=False)
mostrar(tabla_opt)
mejor = tabla_opt.sort_values("mejora_pct", ascending=False).iloc[0]
print(f"📝 La técnica con mayor mejora fue «{mejor.tecnica}» ({mejor.mejora_pct:+.1f} %).")
print("📝 Con ~284 mil filas los tiempos son de segundos; las mejoras crecen con el volumen, que es el escenario real de un banco.")

## Cierre
Respalda el Lakehouse (en Colab) y confirma la ejecución completa.

In [ ]:
respaldar_lake()
print("✅ Notebook", NOTEBOOK, "terminado · run_id:", RUN_ID)